# Data Cleaning 

Nama : Grace Wahyuni 

NIM : 245150401111029

Ini merupakan proses pembersihan data 20 artikel terkait Deep Learning hasil dari scraping pada web Arxiv menggunakan library Selenium 

In [3]:
import pandas as pd

## Load Dataset

In [ ]:
df = pd.read_csv('data/articles.csv', encoding='latin-1')

In [24]:
df.head()

,judul,tahun,penulis,publisher,tipe,kata_kunci,abstrak,pdf_url,thumbnail_path
0,Hybrid Deep Reinforcement Learning for Joint Resource Allocation in Multi-Active RIS-Aided Uplink Communications,2025,"Mohamed Shalma, Engy Aly Maher, Ahmed El-Mahdy",arXiv,Preprint,"Reconfigurable Intelligent Surfaces (RIS), Deep\nReinforcement Learning (DRL), Soft Actor-Critic (SAC), Deep\nDeterministic Policy Gradient (DDPG)","Active Reconfigurable Intelligent Surfaces (RIS) are\na promising technology for 6G wireless networks. This paper\ninvestigates a novel hybrid deep reinforcement learning (DRL)\nframework for resource allocation in a multi-user uplink system\nassisted by multiple active RISs. The objective is to maximize the\nminimum user rate by jointly optimizing user transmit powers,\nactive RIS configurations, and base station (BS) beamforming.\nWe derive a closed-form solution for optimal beamforming\nand employ DRL algorithmsSoft actor-critic (SAC), deep\ndeterministic policy gradient (DDPG), and twin delayed DDPG\n(TD3)to solve the high-dimensional, non-convex power and RIS\noptimization problem. Simulation results demonstrate that SAC\nachieves superior performance with high learning rate leading\nto faster convergence and lower computational cost compared\nto DDPG and TD3. Furthermore, the closed-form of optimally\nbeamforming enhances the minimum rate effectively.",https://arxiv.org/pdf/2512.22107,data/thumbnail/arxiv_1.png
1,Learning continually with representational drift,2025,"Suzanne van der Veldt, Gido M. van de Ven, Sanne Moorman, Guillaume Etter",arXiv,Preprint,"Continual Learning, Representational Drift, Stability-Plasticity Trade-off, Catastrophic Forgetting, Biological Neural Networks, Neuroscience-inspired AI, Non-stationary Data.","Deep artificial neural networks famously struggle to learn from non-stationary streams of data. Without dedicated\nmitigation strategies, continual learning is associated with continuous forgetting of previous tasks and a progressive loss\nof plasticity. Current approaches to continual learning have either focused on increasing the stability of representations\nof past tasks, or on promoting plasticity for future learning. Paradoxically, while animals including humans achieve a\ndesirable stability-plasticity trade-off, the responses of biological neurons to external stimuli that are associated with\nstable behaviors gradually change over time. This suggests that, although unstable representations have historically\nbeen seen as undesirable in artificial systems, they could be a core property of biological neural networks learning\ncontinually. Here, we examine how linking representational drift to continual learning in biological neural networks could\ninform artificial systems. We highlight the existence of representational drift across numerous animal species and brain\nregions and propose that drift reflects a mixture of homeostatic turnover and learning-related synaptic plasticity. In\nparticular, we evaluate how plasticity induced by learning new tasks could induce drift in the representation of previous\ntasks, and how such drift could accumulate across brain regions. In deep artificial neural networks, we propose that\nrepresentational drift is only compatible with approaches that do not explicitly prevent parameter changes to mitigate\nforgetting. Remarkably, jointly promoting plasticity while mitigating forgetting could in principle induce representational\ndrift in continual learning. While we argue that drift is a byproduct rather than a solution to incremental learning, its\ninvestigation could inform approaches to continual learning in artificial systems.",https://arxiv.org/pdf/2512.22045,data/thumbnail/arxiv_2.png
2,Deep Learning Based Auction Design for Selling Agricultural Produce through Farmer Collectives to Maximize Nash Social Welfare,2025,"Mayank Ratan Bhardwaj, Vishisht Srihari Rao, Bazil Ahmed, Kartik Sagar, Y. Narahari",arXiv,Preprint,"Mechanism Design, Volume Dis

## Basic Inspection

In [ ]:
# Display basic info
print("Dataset Shape:", df.shape)
print("\nColumn Names:", df.columns.tolist())
print("\n--- Missing Values Analysis ---")
print("\nMissing Values per Column:")
missing_values = df.isnull().sum()
print(missing_values)

print("\nMissing Values Percentage:")
missing_percentage = (df.isnull().sum() / len(df)) * 100
print(missing_percentage.round(2))

print("\nTotal Missing Values:", df.isnull().sum().sum())

Dataset Shape: (20, 9)

Column Names: ['judul', 'tahun', 'penulis', 'publisher', 'tipe', 'kata_kunci', 'abstrak', 'pdf_url', 'thumbnail_path']

--- Missing Values Analysis ---

Missing Values per Column:
judul             0
tahun             0
penulis           0
publisher         0
tipe              0
kata_kunci        0
abstrak           0
pdf_url           0
thumbnail_path    0
dtype: int64

Missing Values Percentage:
judul             0.0
tahun             0.0
penulis           0.0
publisher         0.0
tipe              0.0
kata_kunci        0.0
abstrak           0.0
pdf_url           0.0
thumbnail_path    0.0
dtype: float64

Total Missing Values: 0


In [10]:
# Check for duplicates
print("--- Duplicate Analysis ---")
print("\nTotal Duplicate Rows:", df.duplicated().sum())
print("\nDuplicates per Column (based on unique values):")
for col in df.columns:
    duplicates = df[col].duplicated().sum()
    print(f"  {col}: {duplicates} duplicates")

# Check for duplicate titles (judul)
print("\n--- Duplicate Titles ---")
duplicate_titles = df[df.duplicated(subset=['judul'], keep=False)]
if len(duplicate_titles) > 0:
    print(f"Found {len(duplicate_titles)} rows with duplicate titles:")
    print(duplicate_titles[['judul', 'tahun']])
else:
    print("No duplicate titles found.")

# Check for duplicate PDF URLs
print("\n--- Duplicate PDF URLs ---")
duplicate_urls = df[df.duplicated(subset=['pdf_url'], keep=False)]
if len(duplicate_urls) > 0:
    print(f"Found {len(duplicate_urls)} rows with duplicate PDF URLs:")
    print(duplicate_urls[['judul', 'pdf_url']])
else:
    print("No duplicate PDF URLs found.")

--- Duplicate Analysis ---

Total Duplicate Rows: 0

Duplicates per Column (based on unique values):
  judul: 0 duplicates
  tahun: 19 duplicates
  penulis: 0 duplicates
  publisher: 19 duplicates
  tipe: 19 duplicates
  kata_kunci: 0 duplicates
  abstrak: 0 duplicates
  pdf_url: 0 duplicates
  thumbnail_path: 0 duplicates

--- Duplicate Titles ---
No duplicate titles found.

--- Duplicate PDF URLs ---
No duplicate PDF URLs found.


## Advanced Inspection 

In [ ]:
import re

# Check for encoding/weird characters
print("--- Encoding/Weird Character Analysis ---\n")

# Common mojibake patterns (encoding errors)
mojibake_patterns = [
    (r'â€™', "apostrophe (')"),
    (r'â€"', 'em dash (—)'),
    (r'â€œ', 'left double quote (")'),
    (r'â€', 'right double quote (")'),
    (r'â€"', 'en dash (–)'),
    (r'Ã©', 'e with accent (é)'),
    (r'Ã¨', 'e with grave (è)'),
    (r'Ã¶', 'o with umlaut (ö)'),
    (r'Ã¼', 'u with umlaut (ü)'),
    (r'â€"s', 'em dash with s'),
    (r'â"', 'quote marks'),
]

def find_weird_chars(text):
    """Find non-ASCII and potentially problematic characters"""
    if pd.isna(text):
        return []
    weird = []
    non_ascii = re.findall(r'[^\x00-\x7F]', str(text))
    if non_ascii:
        weird.extend(set(non_ascii))
    return weird

def find_mojibake(text):
    """Find common mojibake patterns"""
    if pd.isna(text):
        return []
    found = []
    for pattern, desc in mojibake_patterns:
        if re.search(pattern, str(text)):
            found.append(desc)
    return found

# Check each text column
text_columns = ['judul', 'penulis', 'kata_kunci', 'abstrak']

# Collect all rows with special characters
rows_with_issues = []

for idx, row in df.iterrows():
    for col in text_columns:
        value = row[col]
        if pd.isna(value):
            continue
        
        value_str = str(value)
        weird_chars = find_weird_chars(value_str)
        mojibake = find_mojibake(value_str)
        
        if weird_chars or mojibake:
            rows_with_issues.append({
                'row_index': idx,
                'column': col,
                'weird_chars': ''.join(weird_chars) if weird_chars else 'None',
                'mojibake_patterns': ', '.join(mojibake) if mojibake else 'None',
                'text_preview': value_str[:100] + '...' if len(value_str) > 100 else value_str
            })

issues_df = pd.DataFrame(rows_with_issues)

print(f"Total rows with special characters: {len(issues_df)}\n")

if len(issues_df) > 0:
    # Group by column
    print("--- Summary by Column ---")
    print(issues_df.groupby('column').size())
    print()
    
    # Show full table
    print("--- All Rows with Special Characters ---\n")
    
    # Display with settings to show full text
    pd.set_option('display.max_colwidth', None)
    pd.set_option('display.max_rows', None)
    
    for idx, issue_row in issues_df.iterrows():
        print(f"Row {issue_row['row_index']} | Column: {issue_row['column']}")
        print(f"  Weird chars: {issue_row['weird_chars']}")
        print(f"  Mojibake patterns: {issue_row['mojibake_patterns']}")
        print(f"  Text: {issue_row['text_preview']}")
        print()
else:
    print("No special characters or encoding issues found.")

--- Encoding/Weird Character Analysis ---

Total rows with special characters: 9

--- Summary by Column ---
column
abstrak       6
kata_kunci    2
penulis       1
dtype: int64

--- All Rows with Special Characters ---

Row 0 | Column: abstrak
  Weird chars: 
  Mojibake patterns: None
  Text: Active Reconfigurable Intelligent Surfaces (RIS) are
a promising technology for 6G wireless networks...

Row 5 | Column: kata_kunci
  Weird chars: ·
  Mojibake patterns: None
  Text: Dual-Stream Deep Learning · Transformer Encoders · 1D Convolutional Neural Networks · Protein
Langua...

Row 6 | Column: kata_kunci
  Weird chars: 
  Mojibake patterns: None
  Text:  Geomagnetic fields (646)  Cosmic rays (329)  Forbush decreases (546)  Space weather
(2037)  Neu...

Row 6 | Column: abstrak
  Weird chars: 
  Mojibake patterns: None
  Text: Geomagnetic storms (GSTs) driven by solar wind-magnetosphere coupling can severely disrupt technolog...

Row 8 | Column: abstrak
  Weird chars: Åµ
  Mojibake pa

In [18]:
# Check writers/penulis format consistency
print("--- Writers (Penulis) Format Analysis ---\n")

# Show all unique formats
print("Sample of penulis values:\n")
for idx, value in df['penulis'].head(10).items():
    print(f"Row {idx}: {value}")
print()

# Check for different separators
print("--- Separator Analysis ---\n")

def analyze_separator(text):
    """Analyze what separator is used for authors"""
    if pd.isna(text):
        return 'missing'
    text = str(text)
    if ';' in text:
        return 'semicolon (;)'
    elif ',' in text and ' and ' in text.lower():
        return 'comma + and'
    elif ',' in text:
        return 'comma (,)'
    elif ' and ' in text.lower():
        return 'and'
    else:
        return 'single author / other'

df['separator_type'] = df['penulis'].apply(analyze_separator)
print("Separator types found:")
print(df['separator_type'].value_counts())
print()

# Check for quotes around names (might indicate formatting issues)
print("--- Quote Format Analysis ---\n")
quoted_authors = df[df['penulis'].astype(str).str.startswith('"')]
if len(quoted_authors) > 0:
    print(f"Found {len(quoted_authors)} rows with quoted author names:")
    for idx, row in quoted_authors.iterrows():
        print(f"  Row {idx}: {row['penulis']}")
else:
    print("No quoted author names found.")

# Check for inconsistent name formats
print("\n--- Name Format Patterns ---\n")

def check_name_format(text):
    """Check if names follow consistent format"""
    if pd.isna(text):
        return 'missing'
    text = str(text)
    
    # Check for "Last, First" format
    if re.search(r'\w+,\s*\w+', text):
        return 'Last, First format detected'
    # Check for "First Last" format
    elif re.search(r'\w+\s+\w+', text):
        return 'First Last format'
    else:
        return 'single name / other'

df['name_format'] = df['penulis'].apply(check_name_format)
print("Name format patterns:")
print(df['name_format'].value_counts())
print()

# Show rows with potential inconsistencies
print("--- Rows with Potential Issues ---\n")
issues = df[df['separator_type'].isin(['single author / other', 'and']) | 
            df['name_format'].isin(['missing', 'single name / other'])]

if len(issues) > 0:
    print(f"Found {len(issues)} rows with potential format issues:")
    for idx, row in issues.iterrows():
        print(f"  Row {idx}: {row['penulis'][:100]}...")
else:
    print("All author entries follow consistent format.")

# Count authors per paper
print("\n--- Author Count per Paper ---\n")

def count_authors(text):
    """Estimate number of authors"""
    if pd.isna(text):
        return 0
    text = str(text)
    # Split by common separators
    if ';' in text:
        return len([a.strip() for a in text.split(';') if a.strip()])
    elif ',' in text:
        authors = [a.strip() for a in text.split(',') if a.strip()]
        return len(authors)
    elif ' and ' in text.lower():
        return text.lower().count(' and ') + 1
    else:
        return 1

df['author_count'] = df['penulis'].apply(count_authors)
print("Author count distribution:")
print(df['author_count'].value_counts().sort_index())

--- Writers (Penulis) Format Analysis ---

Sample of penulis values:

Row 0: Mohamed Shalma, Engy Aly Maher, Ahmed El-Mahdy
Row 1: Suzanne van der Veldt, Gido M. van de Ven, Sanne Moorman, Guillaume Etter
Row 2: Mayank Ratan Bhardwaj, Vishisht Srihari Rao, Bazil Ahmed, Kartik Sagar, Y. Narahari
Row 3: Nagham Osman, Vittorio Lembo, Giovanni Bottegoni, Laura Toni
Row 4: Chengyu Tian, Wenbin Pei
Row 5: Aicha Boutorh, Soumia Bouyahiaoui, Sara Belhadj, Nour El Yakine Guendouz, Manel Kara Laouar
Row 6: Zongyuan Ge, Chenwaner Zhang, Wei Zhou, Hongyu Zeng, Guiping Zhou
Row 7: Indiwara Nanayakkara, Dehan Jayawickrama, Dasuni Jayawardena, Vijitha R. Herath, Arjuna Madanayake
Row 8: Hao-Yu Zhu, Shi-Jie Du, Lu Xu, Wei Shi
Row 9: Rahul Gupta

--- Separator Analysis ---

Separator types found:
separator_type
comma (,)                17
single author / other     3
Name: count, dtype: int64

--- Quote Format Analysis ---

No quoted author names found.

--- Name Format Patterns ---

Name format pattern

In [21]:
# Check kata_kunci (keywords) separators
print("--- Keywords (Kata Kunci) Separator Analysis ---\n")

# Show sample of keywords
print("Sample of kata_kunci values:\n")
for idx, value in df['kata_kunci'].head(10).items():
    print(f"Row {idx}: {repr(value)}")
print()

# Analyze separators used in keywords
print("--- Separator Types Found ---\n")

def analyze_keyword_separator(text):
    """Analyze what separators are used for keywords"""
    if pd.isna(text):
        return 'missing'
    
    text = str(text)
    separators = []
    
    # Check for common separators
    if ';' in text:
        separators.append('semicolon (;)')
    if ',' in text:
        separators.append('comma (,)')
    if '\n' in text:
        separators.append('newline')
    if '•' in text:
        separators.append('bullet (•)')
    if '|' in text:
        separators.append('pipe (|)')
    if '/' in text:
        separators.append('slash (/)')
    if ' and ' in text.lower():
        separators.append('and')
    
    if len(separators) == 0:
        return 'single keyword / no separator'
    elif len(separators) == 1:
        return separators[0]
    else:
        return ' + '.join(separators)

df['keyword_separator'] = df['kata_kunci'].apply(analyze_keyword_separator)
print("Separator types distribution:")
print(df['keyword_separator'].value_counts())
print()

# Show rows with multiple/unclear separators
print("--- Rows with Multiple Separators ---\n")
multiple_sep = df[df['keyword_separator'].str.contains('\+', na=False)]
if len(multiple_sep) > 0:
    print(f"Found {len(multiple_sep)} rows with multiple separators:")
    for idx, row in multiple_sep.iterrows():
        print(f"\nRow {idx}:")
        print(f"  Separator: {row['keyword_separator']}")
        print(f"  Keywords: {repr(row['kata_kunci'][:150])}")
else:
    print("No rows with multiple separators found.")

# Show unique separator patterns
print("\n--- All Unique Separator Patterns ---\n")
for sep_type in df['keyword_separator'].unique():
    count = (df['keyword_separator'] == sep_type).sum()
    sample = df[df['keyword_separator'] == sep_type]['kata_kunci'].iloc[0]
    print(f"{sep_type}: {count} rows")
    print(f"  Example: {repr(sample[:100])}")
    print()

--- Keywords (Kata Kunci) Separator Analysis ---

Sample of kata_kunci values:

Row 0: 'Reconfigurable Intelligent Surfaces (RIS), Deep\nReinforcement Learning (DRL), Soft Actor-Critic (SAC), Deep\nDeterministic Policy Gradient (DDPG)'
Row 1: 'Continual Learning, Representational Drift, Stability-Plasticity Trade-off, Catastrophic Forgetting, Biological Neural Networks, Neuroscience-inspired AI, Non-stationary Data.'
Row 2: 'Mechanism Design, Volume Discount Auction, Deep Learning for Auctions, Nash Social Welfare, Farmer Collectives, Incentive Compatibility, Agricultural Markets.'
Row 3: 'Hit Identification, De Novo Molecular Design, Generative Models, Hit-like Molecule Generation, Drug Discovery Pipeline, Molecular Diffusion Models, Virtual Screening.'
Row 4: 'Hypergraph isomorphism network, network\nrobustness, hypergraph, prediction.'
Row 5: 'Dual-Stream Deep Learning · Transformer Encoders · 1D Convolutional Neural Networks · Protein\nLanguage Models (ESM-2) · Antigen-Antibody Int

## Key findings                                                                                                           
                                                                                                                                   
  1. Missing Values                                                                                                            
    Tidak ada missing value

  2. Duplicates 
    Tidak ada duplikat baris, judul, maupun url pdf

  3. Encoding Issues 
    Found 9 rows with special characters/mojibake, in rows abstrak, kata_kunci, and penulis

  4. Writers Format 
    Format is consistent 

  5. Keywords Separator 
    There is an inconsistency in separator, which is :
  - Comma + newline: 8 rows
  - Comma only: 8 rows
  - Newline only (bullet ·): 2 rows
  - Semicolon: 2 rows

## Data Cleaning

In [25]:
import re
def fix_encoding(text):
    """Fix mojibake and special characters"""
    if pd.isna(text):
        return text
    
    text = str(text)
    
    # Fix common mojibake patterns
    replacements = {
        'â€™': "'",      # apostrophe
        'â€"': '—',     # em dash
        'â€"': '—',     # em dash (alternate)
        'â€œ': '"',     # left double quote
        'â€': '"',      # right double quote
        'â€"': '–',     # en dash
        'Ã©': 'é',      # e with accent
        'Ã¨': 'è',      # e with grave
        'Ã¶': 'ö',      # o with umlaut
        'Ã¼': 'ü',      # u with umlaut
        'â€"s': '—s',   # em dash with s
        'â"': '"',     # quote marks
    }
    
    for wrong, correct in replacements.items():
        text = text.replace(wrong, correct)
    
    # Fix remaining special characters
    text = text.replace('—', '-')    
    text = text.replace('–', '-')    
    text = text.replace('"', '"')   
    text = text.replace('"', '"')   
    text = text.replace(''', "'") 
    text = text.replace(''', "'")
    text = text.replace('·', ',')   
    text = text.replace('×', 'x')   
    text = text.replace('Å', 'A')  
    
    return text

# Apply encoding fix to text columns
text_columns = ['judul', 'penulis', 'kata_kunci', 'abstrak']

print("Fixing encoding issues...")
for col in text_columns:
    df[col] = df[col].apply(fix_encoding)

print("✓ Encoding issues fixed!")

# Verify fixes
print("\n--- Verification ---")
issues_after = []
for idx, row in df.iterrows():
    for col in text_columns:
        value = str(row[col])
        weird = re.findall(r'[^\x00-\x7F]', value)
        if weird:
            issues_after.append({
                'row': idx,
                'column': col,
                'chars': ''.join(set(weird))
            })

if len(issues_after) > 0:
    print(f"Remaining issues: {len(issues_after)}")
    for issue in issues_after[:5]:
        print(f"  Row {issue['row']} ({issue['column']}): {issue['chars']}")
else:
    print("✓ No more encoding issues found!")

Fixing encoding issues...
✓ Encoding issues fixed!

--- Verification ---
Remaining issues: 7
  Row 0 (abstrak): 
  Row 6 (kata_kunci): 
  Row 6 (abstrak): 
  Row 8 (abstrak): µ
  Row 13 (penulis): é


In [26]:
def standardize_keywords(text):
    """Standardize all keywords to use semicolon separator"""
    if pd.isna(text):
        return text
    
    text = str(text)
    
    # Replace newlines with space
    text = text.replace('\n', ' ')
    
    # Replace bullet points with semicolon
    text = text.replace('·', ';')
    text = text.replace('•', ';')
    
    # Handle semicolons that might have been created
    # First, normalize multiple spaces
    text = re.sub(r'\s+', ' ', text)
    
    # Replace ", " with "; " for comma-separated keywords
    # But be careful with abbreviations like "RIS), Deep"
    # Split by comma and rejoin with semicolon
    if ';' not in text:  # Only process if not already using semicolon
        # Split by comma, but handle special cases
        parts = [p.strip() for p in text.split(',')]
        # Rejoin with semicolon
        text = '; '.join(parts)
    else:
        # Already has semicolon, just clean up spacing
        text = re.sub(r'\s*;\s*', '; ', text)
    
    # Clean up any double semicolons
    text = re.sub(r';\s*;', ';', text)
    
    # Remove trailing punctuation
    text = text.strip().rstrip('.')
    
    return text

print("Standardizing keyword separators...")
print("\n--- Before ---")
print(df['kata_kunci'].head(5).to_string())

# Apply standardization
df['kata_kunci'] = df['kata_kunci'].apply(standardize_keywords)

print("\n--- After ---")
print(df['kata_kunci'].head(5).to_string())

# Verify all use semicolon now
print("\n--- Verification ---")
semicolon_count = df['kata_kunci'].str.contains(';').sum()
print(f"Rows using semicolon: {semicolon_count}/{len(df)}")

# Show all unique separator patterns after cleaning
def check_separator(text):
    if pd.isna(text):
        return 'missing'
    text = str(text)
    if '\n' in text:
        return 'newline'
    elif ';' in text:
        return 'semicolon'
    elif ',' in text:
        return 'comma'
    else:
        return 'single keyword'

df['sep_after'] = df['kata_kunci'].apply(check_separator)
print("\nSeparator distribution after cleaning:")
print(df['sep_after'].value_counts())
df.drop('sep_after', axis=1, inplace=True)

print("\n✓ Keyword separators standardized to semicolon!")

Standardizing keyword separators...

--- Before ---
0                                 Reconfigurable Intelligent Surfaces (RIS), Deep\nReinforcement Learning (DRL), Soft Actor-Critic (SAC), Deep\nDeterministic Policy Gradient (DDPG)
1    Continual Learning, Representational Drift, Stability-Plasticity Trade-off, Catastrophic Forgetting, Biological Neural Networks, Neuroscience-inspired AI, Non-stationary Data.
2                     Mechanism Design, Volume Discount Auction, Deep Learning for Auctions, Nash Social Welfare, Farmer Collectives, Incentive Compatibility, Agricultural Markets.
3             Hit Identification, De Novo Molecular Design, Generative Models, Hit-like Molecule Generation, Drug Discovery Pipeline, Molecular Diffusion Models, Virtual Screening.
4                                                                                                       Hypergraph isomorphism network, network\nrobustness, hypergraph, prediction.

--- After ---
0                           

In [27]:
# Display final cleaned dataset info
print("=" * 60)
print("CLEANING COMPLETE")
print("=" * 60)

print("\n--- Final Dataset Info ---")
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(f"\nColumns: {df.columns.tolist()}")

print("\n--- Sample of Cleaned Data ---")
print("\nTitles (judul):")
print(df['judul'].head())

print("\n\nKeywords (kata_kunci) - first 3 entries:")
for idx, kw in df['kata_kunci'].head(3).items():
    print(f"\nRow {idx}: {kw[:100]}...")

print("\n\nAuthors (penulis) - first 3 entries:")
for idx, author in df['penulis'].head(3).items():
    print(f"Row {idx}: {author}")

# Save cleaned data
output_path = 'data/articles_cleaned.csv'
df.to_csv(output_path, index=False, encoding='utf-8')
print(f"\n\n✓ Cleaned data saved to: {output_path}")

CLEANING COMPLETE

--- Final Dataset Info ---
Total rows: 20
Total columns: 9

Columns: ['judul', 'tahun', 'penulis', 'publisher', 'tipe', 'kata_kunci', 'abstrak', 'pdf_url', 'thumbnail_path']

--- Sample of Cleaned Data ---

Titles (judul):
0                                                       Hybrid Deep Reinforcement Learning for Joint Resource Allocation in Multi-Active RIS-Aided Uplink Communications
1                                                                                                                       Learning continually with representational drift
2                                         Deep Learning Based Auction Design for Selling Agricultural Produce through Farmer Collectives to Maximize Nash Social Welfare
3                                                                                   From In Silico to In Vitro: Evaluating Molecule Generative Models for Hit Generation
4    HWL-HIN: A Hypergraph-Level Hypergraph Isomorphism Network as Powerful as the